<a href="https://colab.research.google.com/github/ParhamPishro/Flavonoid/blob/main/7%20Final%20Model%20and%20Evaluation/Classification%20Reports.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import cm

from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_val_predict, cross_val_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

Read Data

In [2]:
df = pd.read_excel("/content/FlavonoidData.xlsx")
df

,authors,flavonoid,cell line,time,dose,viability-mean,viability-error,viability-SD,viability-SE
0,Chen et al. 2017,Luteolin,KYSE30,48,10.0,77.272727,77.272727,0.001000,NaN
1,Chen et al. 2017,Luteolin,KYSE30,48,20.0,67.613636,67.613636,0.001000,NaN
2,Chen et al. 2017,Luteolin,KYSE30,48,40.0,47.159091,47.159091,0.001000,NaN
3,Chen et al. 2017,Luteolin,KYSE30,48,80.0,32.386364,32.386364,0.001000,NaN
4,Chen et al. 2017,Luteolin,KYSE30,72,10.0,69.886364,69.886364,0.001000,NaN
...,...,...,...,...,...,...,...,...,...
479,Zhu et al. 2016,Apigenin,KYSE150,24,20.0,88.407643,88.407643,0.001000,NaN
480,Zhu et al. 2016,Apigenin,KYSE150,24,40.0,78.980892,82.802548,3.821656,NaN
481,Zhu et al. 2016,Apigenin,KYSE150,24,60.0,67.261146,70.828025,3.566879,NaN
482,Zhu et al. 2016,Apigenin,KYSE150,24,80.0,45.350318,47.643312,2.292994,NaN


In [59]:
test = pd.read_excel("/content/FlavonoidTest.xlsx")

Discretized Data

In [60]:
df['v01'] = 1
df.loc[df['viability-mean']>50, 'v01'] = 0

test['v01'] = 1
test.loc[test['viability-mean']>50, 'v01'] = 0

In [4]:
df = df.drop(['authors', 'viability-error', 'viability-SD', 'viability-SE'], axis=1)
df

,flavonoid,cell line,time,dose,viability-mean,v01
0,Luteolin,KYSE30,48,10.0,77.272727,0
1,Luteolin,KYSE30,48,20.0,67.613636,0
2,Luteolin,KYSE30,48,40.0,47.159091,1
3,Luteolin,KYSE30,48,80.0,32.386364,1
4,Luteolin,KYSE30,72,10.0,69.886364,0
...,...,...,...,...,...,...
479,Apigenin,KYSE150,24,20.0,88.407643,0
480,Apigenin,KYSE150,24,40.0,78.980892,0
481,Apigenin,KYSE150,24,60.0,67.261146,0
482,Apigenin,KYSE150,24,80.0,45.350318,1


Encoding

In [5]:
df_encoded = pd.get_dummies(df, dtype=int)
df_encoded

,time,dose,viability-mean,v01,flavonoid_Acacetin,flavonoid_Apigenin,flavonoid_Chrysin,flavonoid_Galangin,flavonoid_Kaempferol,flavonoid_Luteolin,flavonoid_Myricetin,flavonoid_Quercetin,cell line_KYSE150,cell line_KYSE30,cell line_KYSE410,cell line_KYSE450,cell line_KYSE510,cell line_OE19,cell line_OE33,cell line_TE-1,cell line_TE-10,cell line_YES2
0,48,10.0,77.272727,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0
1,48,20.0,67.613636,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0
2,48,40.0,47.159091,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0
3,48,80.0,32.386364,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0
4,72,10.0,69.886364,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479,24,20.0,88.407643,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
480,24,40.0,78.980892,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
481,24,60.0,67.261146,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
482,24,80.0,45.350318,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0


In [61]:
X = df_encoded.drop(['viability-mean', 'v01'], axis=1)
y = df['viability-mean']
y01 = df['v01']

X_test = test.drop(['authors', 'flavonoid', 'cell line', 'viability-mean', 'v01'], axis=1)
y_test = test['v01']

Initialize

In [46]:
knc = KNeighborsClassifier(metric='euclidean', n_neighbors=51, weights='distance')
lor = LogisticRegression(C=0.5, penalty='l1', solver='liblinear')
svc = SVC(C=2**10, gamma=0.015625, kernel='rbf')
dtc = DecisionTreeClassifier(criterion='gini', max_depth=14, min_samples_leaf=1, min_samples_split=2, random_state=2, splitter='best')
rfc = RandomForestClassifier(criterion='gini', max_depth=11, min_samples_leaf=1, min_samples_split=2, n_estimators=5, random_state=2)
ada = AdaBoostClassifier(learning_rate=1, n_estimators=15)
xgb = XGBClassifier(learning_rate=1, n_estimators=10)

kf = KFold(n_splits=5, shuffle=False)


Simple Model

In [42]:
X_simple = X[['time', 'dose']]
X_simple

,time,dose
0,48,10.0
1,48,20.0
2,48,40.0
3,48,80.0
4,72,10.0
...,...,...
479,24,20.0
480,24,40.0
481,24,60.0
482,24,80.0


In [47]:
param_kn = {'n_neighbors': range(3, 100, 2),
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan', 'minkowski']}
gs_kn = GridSearchCV(knc, param_kn, cv=5, scoring='accuracy')
%time gs_kn.fit(X_simple, y01)
print(gs_kn.best_params_)
print(gs_kn.best_score_)

CPU times: user 6.69 s, sys: 11.1 ms, total: 6.7 s
Wall time: 6.73 s
{'metric': 'euclidean', 'n_neighbors': 21, 'weights': 'uniform'}
0.8305412371134022


In [48]:
param_lo = {'penalty': ['l1', 'l2'],
            'C': [2**_ for _ in range(-15,11)],
            'solver': ['liblinear']}

gs_lo = GridSearchCV(lor, param_lo, cv=5, scoring='accuracy')
%time gs_lo.fit(X_simple, y01)
print(gs_lo.best_params_)
print(gs_lo.best_score_)

CPU times: user 929 ms, sys: 1.05 ms, total: 930 ms
Wall time: 930 ms
{'C': 0.5, 'penalty': 'l1', 'solver': 'liblinear'}
0.814024914089347


In [49]:
param_dt = {'criterion': ['gini', 'entropy'],
         'splitter': ['best'],
         'max_depth': range(1, 21),
         'min_samples_split': [2],
         'min_samples_leaf': [1],
         'random_state': [2]}

gs_dt = GridSearchCV(dtc, param_dt, cv = 5, scoring = "accuracy")
%time gs_dt.fit(X_simple, y01)
print(gs_dt.best_params_)
print(gs_dt.best_score_)

CPU times: user 687 ms, sys: 4.1 ms, total: 692 ms
Wall time: 692 ms
{'criterion': 'gini', 'max_depth': 6, 'min_samples_leaf': 1, 'min_samples_split': 2, 'random_state': 2, 'splitter': 'best'}
0.824355670103093


In [50]:
param_rf = {'criterion': ['gini', 'entropy'],
         'max_depth': range(1, 21),
         'min_samples_split': [2],
         'min_samples_leaf': [1],
         'n_estimators': range(5, 101, 5),
         'random_state': [2]}

gs_rf = GridSearchCV(rfc, param_rf, cv = 5, scoring = "accuracy")
%time gs_rf.fit(X_simple, y01)
print(gs_rf.best_params_)
print(gs_rf.best_score_)

CPU times: user 4min 1s, sys: 628 ms, total: 4min 1s
Wall time: 4min 2s
{'criterion': 'gini', 'max_depth': 1, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 15, 'random_state': 2}
0.826417525773196


In [ ]:
param_ad = {'n_estimators': range(5, 105, 5),
            'learning_rate': [0.0001, 0.001, 0.01, 0.1, 1, 10 ,100, 1000, 10000]}
gs_ad = GridSearchCV(ada, param_ad, cv=5, scoring='accuracy')
%time gs_ad.fit(X_simple, y01)
print(gs_ad.best_params_)
print(gs_ad.best_score_)

In [52]:
param_xg = {'n_estimators': range(5, 105, 5),
            'learning_rate': [0.0001, 0.001, 0.01, 0.1, 1, 10 ,100, 1000, 10000]}
gs_xg = GridSearchCV(xgb, param_xg, cv=5, scoring='accuracy')
%time gs_xg.fit(X_simple, y01)
print(gs_xg.best_params_)
print(gs_xg.best_score_)

CPU times: user 24.8 s, sys: 621 ms, total: 25.4 s
Wall time: 13.5 s
{'learning_rate': 0.1, 'n_estimators': 85}
0.8284793814432991


In [53]:
#initialize

knc = KNeighborsClassifier(metric='euclidean', n_neighbors=21, weights='uniform')
lor = LogisticRegression(C=0.5, penalty='l1', solver='liblinear')
svc = SVC(C=2, gamma=0.0009765625, kernel='rbf')
dtc = DecisionTreeClassifier(criterion='gini', max_depth=6, min_samples_leaf=1, min_samples_split=2, random_state=2, splitter='best')
rfc = RandomForestClassifier(criterion='gini', max_depth=1, min_samples_leaf=1, min_samples_split=2, n_estimators=15, random_state=2)
ada = AdaBoostClassifier(learning_rate=1, n_estimators=10)
xgb = XGBClassifier(learning_rate=0.1, n_estimators=85)


In [54]:
kf_knc = cross_val_score(knc, X_simple, y01, cv=kf, scoring='accuracy')
print(kf_knc.mean(), kf_knc.std())

0.8304553264604811 0.049615094212526044


In [55]:
dt_knc = cross_val_score(dtc, X_simple, y01, cv=kf, scoring='accuracy')
print(dt_knc.mean(), dt_knc.std())

0.8201460481099657 0.0654213429578286


In [56]:
#fit

knc.fit(X_simple, y01)
svc.fit(X_simple, y01)
lor.fit(X_simple, y01)
dtc.fit(X_simple, y01)
rfc.fit(X_simple, y01)
ada.fit(X_simple, y01)
xgb.fit(X_simple, y01)

# validation

knc_val = knc.predict(X_simple)
lor_val = lor.predict(X_simple)
svc_val = svc.predict(X_simple)
dtc_val = dtc.predict(X_simple)
rfc_val = rfc.predict(X_simple)
ada_val = ada.predict(X_simple)
xgb_val = xgb.predict(X_simple)


In [57]:
# Evaluation

print("KNN : ", accuracy_score(y01, knc_val))
print("LoR : ", accuracy_score(y01, lor_val))#, "\t(Test): ", accuracy_score(y_test, lor_pred))
print("SVM : ", accuracy_score(y01, svc_val))#, "\t(Test): ", accuracy_score(y_test, svc_pred))
print("DT  : ", accuracy_score(y01, dtc_val))#, "\t(Test): ", accuracy_score(y_test, dtc_pred))
print("RF  : ", accuracy_score(y01, rfc_val))#, "\t(Test): ", accuracy_score(y_test, rfc_pred))
print("Ada : ", accuracy_score(y01, ada_val))
print("XGB : ", accuracy_score(y01, xgb_val))

KNN :  0.8326446280991735
LoR :  0.8161157024793388
SVM :  0.8347107438016529
DT  :  0.8512396694214877
RF  :  0.8305785123966942
Ada :  0.8347107438016529
XGB :  0.8491735537190083


In [62]:
# prediction

knc_pred = knc.predict(X_test)
lor_pred = lor.predict(X_test)
svc_pred = svc.predict(X_test)
dtc_pred = dtc.predict(X_test)
rfc_pred = rfc.predict(X_test)
ada_pred = ada.predict(X_test)
xgb_pred = xgb.predict(X_test)


In [66]:
print('KNN', classification_report(y_test, knc_pred))
print('LOR', classification_report(y_test, lor_pred))
print('SVM', classification_report(y_test, svc_pred))
print('DT', classification_report(y_test, dtc_pred))
print('RF', classification_report(y_test, rfc_pred))
print('Ada', classification_report(y_test, ada_pred))
print('XGB', classification_report(y_test, xgb_pred))


KNN               precision    recall  f1-score   support

           0       0.94      0.92      0.93        85
           1       0.65      0.72      0.68        18

    accuracy                           0.88       103
   macro avg       0.79      0.82      0.81       103
weighted avg       0.89      0.88      0.89       103

LOR               precision    recall  f1-score   support

           0       0.91      0.94      0.92        85
           1       0.67      0.56      0.61        18

    accuracy                           0.87       103
   macro avg       0.79      0.75      0.77       103
weighted avg       0.87      0.87      0.87       103

SVM               precision    recall  f1-score   support

           0       0.94      0.92      0.93        85
           1       0.65      0.72      0.68        18

    accuracy                           0.88       103
   macro avg       0.79      0.82      0.81       103
weighted avg       0.89      0.88      0.89       103

DT     